# 🛡️ GraphGuard: Google Drive + Colab Retraining Pipeline

This notebook runs the **GraphGuard (TempoStruct-GAN + GraphSAGE)** training directly inside your **Google Drive**.
- ✅ **Permanent Storage**: All models, plots, and logs save straight to your Google Drive in real-time.
- ✅ **No Data Loss**: Even if Colab disconnects, your trained checkpoints and results remain safely saved in Drive.

---

### Step 1: Set GPU Runtime
> Go to **Runtime** ➔ **Change runtime type** ➔ Select **T4 GPU** ➔ Click **Save**.

In [ ]:
!nvidia-smi

### Step 2: Mount Google Drive
Run this cell and click **Connect to Google Drive** when prompted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 3: Setup Project Directory in Google Drive

**Instructions:**
1. Put `graphguard_colab.zip` into your **Google Drive** (e.g. root `MyDrive/` or inside a folder `MyDrive/GraphGuard/`).
2. Run the cell below — it will automatically detect the zip, extract it into `/content/drive/MyDrive/GraphGuard`, and set your working directory.

In [ ]:
import os
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/GraphGuard')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

# Possible zip locations
possible_zips = [
    '/content/drive/MyDrive/GraphGuard/graphguard_colab.zip',
    '/content/drive/MyDrive/graphguard_colab.zip',
    '/content/graphguard_colab.zip'
]

zip_found = None
for z in possible_zips:
    if os.path.exists(z):
        zip_found = z
        break

if zip_found:
    print(f"Found archive: {zip_found}")
    print(f"Extracting into {DRIVE_DIR} ...")
    !unzip -q -o "{zip_found}" -d "{DRIVE_DIR}"
    print("Extraction complete!")
else:
    if (DRIVE_DIR / 'scripts' / '03_train_classifier.py').exists():
        print(f"Project files already present in {DRIVE_DIR} — ready to train!")
    else:
        print("⚠️ Could not find graphguard_colab.zip!")
        print("Please upload graphguard_colab.zip to your Google Drive (MyDrive) and re-run this cell.")

%cd /content/drive/MyDrive/GraphGuard
!ls -la

### Step 4: Install Dependencies

In [ ]:
%cd /content/drive/MyDrive/GraphGuard
!pip install -q scikit-learn matplotlib seaborn

### Step 5: Train Generator & Edge Predictor (TempoStruct-GAN)
Trains the temporal generator and edge predictor for 100 epochs. Checkpoints save directly to your Google Drive (`graphguard/models/`).

In [ ]:
%cd /content/drive/MyDrive/GraphGuard
!python scripts/02_train_generator.py

### Step 6: Train Downstream GraphSAGE Classifier
Balances the graph with synthetic illicit nodes, injects edges, and trains GraphSAGE for 1000 epochs.
Restores the best checkpoint based on validation F1 and evaluates on the test split.

In [ ]:
%cd /content/drive/MyDrive/GraphGuard
!python scripts/03_train_classifier.py

### Step 7: View Final Test Metrics & Training Curves
Displays the evaluation metrics and loss/F1 curves saved in your Google Drive.

In [ ]:
import glob, json
from IPython.display import Image, display

runs = sorted(glob.glob('/content/drive/MyDrive/GraphGuard/results/classifier_gnn/run_*'))
if runs:
    latest_run = runs[-1]
    print(f"Latest Run: {latest_run}\n")
    
    with open(f"{latest_run}/metrics.json") as f:
        metrics = json.load(f)
        print("=" * 40)
        print("FINAL TEST METRICS:")
        print("=" * 40)
        for k, v in metrics.items():
            print(f"{k:25s}: {v}")
        print("=" * 40)
        
    print("\n--- Validation F1 Curve ---")
    display(Image(f"{latest_run}/val_f1_curve.png"))
    print("\n--- Loss Curve ---")
    display(Image(f"{latest_run}/loss_curve.png"))
else:
    print("No classifier runs found yet. Run Step 6 first!")